# 02 · Process every take

One streaming pass per take, one take at a time (keeps memory at a few hundred MB even for multi-GB files).
Up-to-date bundles are skipped, so re-running is cheap. Use `force=True` after changing the DSP settings
if you want to be sure, although changed settings are detected automatically.

Command-line equivalent (useful for long runs, e.g. overnight):

```bash
python -m zoom_acoustics process config/local_myexperiment.yaml
```

In [1]:
# ==== YOUR SETTINGS ====
CONFIG = "../config/demo.yaml"
ONLY = None          # e.g. ["DEMO_002", "DEMO_003"] to process a subset
FORCE = False

In [2]:
# ---- standard setup (same in every notebook) ----
%matplotlib inline
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import zoom_acoustics as za
from zoom_acoustics import plots as zp, dsp

cfg = za.load_config(CONFIG)
print("config   :", cfg.config_file)
print("data_dir :", cfg.data_dir)
print("cache_dir:", cfg.cache_dir)
print("figures  :", cfg.figures_dir)

config   : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/config/demo.yaml
data_dir : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/audio
cache_dir: /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache
figures  : /media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/figures


In [3]:
takes = za.discover_takes(cfg.data_dir, recursive=cfg.recursive, pattern=cfg.file_pattern)
if ONLY:
    takes = [za.find_take(takes, n) for n in ONLY]
za.process_all(takes, cfg, force=FORCE)

[1/4] DEMO_001  1.0 min, 4 ch, 48000 Hz
DEMO_001: cached bundle is up to date (/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_001)
[2/4] DEMO_002  3.0 min, 4 ch, 48000 Hz
DEMO_002: cached bundle is up to date (/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_002)
[3/4] DEMO_003  2.5 min, 2 ch, 48000 Hz
DEMO_003: cached bundle is up to date (/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_003)
[4/4] DEMO_004  2.0 min, 1 ch, 48000 Hz
DEMO_004: cached bundle is up to date (/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_004)


[PosixPath('/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_001'),
 PosixPath('/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_002'),
 PosixPath('/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_003'),
 PosixPath('/media/tmittal/Disk1/BKG_Data_Drive/zoom_acoustics_toolkit/demo_data/cache/DEMO_004')]

## Summary of every processed take

In [4]:
feats = za.load_all_features(cfg)
za.timeline.takes_overview(feats)

,take,start,end,duration_min,sr,channels,n_triggers
0,DEMO_001,2026-10-01 10:00:00,2026-10-01 10:01:00,1.0,48000,"1,2,3,4",0
1,DEMO_002,2026-10-01 10:02:00,2026-10-01 10:05:00,3.0,48000,"1,2,3,4",6227
2,DEMO_003,2026-10-01 10:06:00,2026-10-01 10:08:30,2.5,48000,"1,2",4255
3,DEMO_004,2026-10-01 10:10:00,2026-10-01 10:12:00,2.0,48000,1,857


In [5]:
summary = pd.concat([f.summary() for f in feats], ignore_index=True)
summary.to_csv(cfg.figures_path / "summary_all_takes.csv", index=False)
summary

,take,channel,label,sensor,peak_abs,low_median_dB,audio_median_dB,rig_median_dB,broadband_median_dB,n_trig_audio,trig_rate_audio_per_s
0,DEMO_001,1,Ch1 Hydrophone A,hydrophone,0.020599,-92.916372,-72.707881,-73.450273,-72.598198,0,0.000000
1,DEMO_001,2,Ch2 Hydrophone B,hydrophone,0.020591,-92.949052,-72.709157,-73.455958,-72.600266,0,0.000000
2,DEMO_001,3,Ch3 Contact sensor,contact,0.010694,-92.990586,-72.700763,-73.442078,-72.589963,0,0.000000
3,DEMO_001,4,Ch4 Air mic,air,0.300590,-92.988519,-72.698986,-73.440119,-72.589818,0,0.000000
4,DEMO_002,1,Ch1 Hydrophone A,hydrophone,0.379766,-78.345387,-61.964275,-71.503594,-61.862493,2893,19.321445
5,DEMO_002,2,Ch2 Hydrophone B,hydrophone,0.380005,-83.870696,-66.747307,-72.834567,-66.626786,2713,18.119281
6,DEMO_002,3,Ch3 Contact sensor,contact,0.379564,-91.872022,-71.931663,-73.393155,-71.831612,621,4.147465
7,DEMO_002,4,Ch4 Air mic,air,0.379784,-92.929884,-72.685328,-73.445751,-72.577609,0,0.000000
8,DEMO_003,1,Ch1 Hydrophone A,hydrophone,0.051958,-80.760920,-64.098065,-72.175156,-63.988213,2174,14.496233
9,DEMO_003,2,Ch2 Hydrophone B,hydrophone,0.026047,-86.061700,-68.187376,-72.998553,-68.091164,2081,13.876109
